# Generazione condizionata di immagini con cWGAN-GP

Il notebook implementa una Conditional Wasserstein GAN con Gradient Penalty sul dataset STL-10.

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt

In [ ]:
# Parametri principali
batch_size = 64
latent_dim = 100
embedding_dim = 50
num_classes = 10
image_size = 64
channels = 3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo utilizzato:", device)

## Preparazione del dataset

Le immagini vengono ridimensionate a 64x64 pixel e normalizzate nell'intervallo [-1, 1].

In [ ]:
transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset = datasets.STL10(
    root="./data",
    split="train",
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

print("Numero di immagini:", len(train_dataset))
print("Numero di batch:", len(train_loader))

In [ ]:
# Visualizzazione di alcune immagini reali
real_images, real_labels = next(iter(train_loader))
grid = make_grid(real_images[:16], nrow=4, normalize=True)

plt.figure(figsize=(6, 6))
plt.imshow(grid.permute(1, 2, 0))
plt.axis("off")
plt.title("Esempi dal dataset STL-10")
plt.show()

print("Forma del batch:", real_images.shape)
print("Intervallo valori:", real_images.min().item(), real_images.max().item())

## Generatore

Il rumore casuale viene unito all'embedding della classe. La rete aumenta gradualmente la risoluzione fino a produrre un'immagine RGB di 64x64 pixel.

In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.label_embedding = nn.Embedding(num_classes, embedding_dim)
        self.input_layer = nn.Linear(latent_dim + embedding_dim, 512 * 4 * 4)

        self.network = nn.Sequential(
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            nn.ConvTranspose2d(64, channels, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, noise, labels):
        label_vectors = self.label_embedding(labels)
        x = torch.cat((noise, label_vectors), dim=1)
        x = self.input_layer(x)
        x = x.view(-1, 512, 4, 4)
        return self.network(x)

## Critico

L'etichetta viene trasformata in una mappa e concatenata all'immagine. L'uscita è un valore scalare senza funzione sigmoid.

In [ ]:
class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.label_embedding = nn.Embedding(num_classes, image_size * image_size)

        self.network = nn.Sequential(
            nn.Conv2d(channels + 1, 64, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, 4, 2, 1),
            nn.LayerNorm([128, 16, 16]),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 4, 2, 1),
            nn.LayerNorm([256, 8, 8]),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 512, 4, 2, 1),
            nn.LayerNorm([512, 4, 4]),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 1, 4, 1, 0)
        )

    def forward(self, images, labels):
        label_maps = self.label_embedding(labels)
        label_maps = label_maps.view(-1, 1, image_size, image_size)
        x = torch.cat((images, label_maps), dim=1)
        return self.network(x).view(-1)

## Controllo delle dimensioni

Prima del training verifichiamo che le due reti restituiscano output con le dimensioni corrette.

In [ ]:
generator = Generator().to(device)
critic = Critic().to(device)

test_noise = torch.randn(8, latent_dim, device=device)
test_labels = torch.arange(8, device=device) % num_classes

with torch.no_grad():
    fake_images = generator(test_noise, test_labels)
    critic_scores = critic(fake_images, test_labels)

print("Output Generatore:", fake_images.shape)
print("Output Critico:", critic_scores.shape)

assert fake_images.shape == (8, 3, 64, 64)
assert critic_scores.shape == (8,)
print("Controllo completato correttamente.")

# Step 2 - Gradient Penalty e ciclo di training

La funzione seguente calcola la penalità del gradiente sui campioni interpolati tra immagini reali e generate.

In [ ]:
def gradient_penalty(critic, real_images, fake_images, labels):
    batch = real_images.size(0)
    alpha = torch.rand(batch, 1, 1, 1, device=device)
    interpolated = alpha * real_images + (1 - alpha) * fake_images
    interpolated.requires_grad_(True)

    scores = critic(interpolated, labels)
    gradients = torch.autograd.grad(
        outputs=scores,
        inputs=interpolated,
        grad_outputs=torch.ones_like(scores),
        create_graph=True,
        retain_graph=True
    )[0]

    gradients = gradients.view(batch, -1)
    return ((gradients.norm(2, dim=1) - 1) ** 2).mean()

## Iperparametri e ottimizzatori

Il Critico viene aggiornato cinque volte per ogni aggiornamento del Generatore.

In [ ]:
learning_rate = 0.0001
lambda_gp = 10
critic_iterations = 5

optimizer_generator = torch.optim.Adam(
    generator.parameters(), lr=learning_rate, betas=(0.0, 0.9)
)
optimizer_critic = torch.optim.Adam(
    critic.parameters(), lr=learning_rate, betas=(0.0, 0.9)
)

## Galleria delle classi e ciclo di addestramento

La galleria usa sempre lo stesso rumore per rendere confrontabili le immagini prodotte nelle diverse epoche. Ogni riga corrisponde a una classe di STL-10.

In [ ]:
class_names = [
    "aereo", "uccello", "auto", "gatto", "cervo",
    "cane", "cavallo", "scimmia", "nave", "camion"
]
samples_per_class = 5
fixed_noise = torch.randn(num_classes * samples_per_class, latent_dim, device=device)
fixed_labels = torch.arange(num_classes, device=device).repeat_interleave(samples_per_class)
os.makedirs("risultati", exist_ok=True)

def show_gallery(epoch):
    generator.eval()
    with torch.no_grad():
        images = generator(fixed_noise, fixed_labels).cpu()

    grid = make_grid(images, nrow=samples_per_class, normalize=True)
    plt.figure(figsize=(7, 13))
    plt.imshow(grid.permute(1, 2, 0))
    plt.axis("off")
    plt.title(f"Galleria delle classi - Epoca {epoch}")
    plt.tight_layout()
    plt.savefig(f"risultati/galleria_epoca_{epoch}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Ordine delle righe:", ", ".join(class_names))
    generator.train()

def train(epochs):
    generator_losses = []
    critic_losses = []

    generator.train()
    critic.train()
    show_gallery(0)

    for epoch in range(epochs):
        generator_loss_sum = 0.0
        critic_loss_sum = 0.0

        for real_images, labels in train_loader:
            real_images = real_images.to(device)
            labels = labels.to(device)
            current_batch = real_images.size(0)

            for _ in range(critic_iterations):
                noise = torch.randn(current_batch, latent_dim, device=device)
                fake_images = generator(noise, labels).detach()

                real_scores = critic(real_images, labels)
                fake_scores = critic(fake_images, labels)
                gp = gradient_penalty(critic, real_images, fake_images, labels)
                critic_loss = fake_scores.mean() - real_scores.mean() + lambda_gp * gp

                optimizer_critic.zero_grad()
                critic_loss.backward()
                optimizer_critic.step()

            noise = torch.randn(current_batch, latent_dim, device=device)
            fake_images = generator(noise, labels)
            generator_loss = -critic(fake_images, labels).mean()

            optimizer_generator.zero_grad()
            generator_loss.backward()
            optimizer_generator.step()

            generator_loss_sum += generator_loss.item()
            critic_loss_sum += critic_loss.item()

        mean_generator_loss = generator_loss_sum / len(train_loader)
        mean_critic_loss = critic_loss_sum / len(train_loader)
        generator_losses.append(mean_generator_loss)
        critic_losses.append(mean_critic_loss)

        print(
            f"Epoca {epoch + 1}/{epochs} - "
            f"Loss Critico: {mean_critic_loss:.4f} - "
            f"Loss Generatore: {mean_generator_loss:.4f}"
        )

        current_epoch = epoch + 1
        if current_epoch % 10 == 0 or current_epoch == epochs // 2:
            show_gallery(current_epoch)

    return generator_losses, critic_losses

In [ ]:
# Addestramento completo
num_epochs = 50
generator_losses, critic_losses = train(epochs=num_epochs)
print("Addestramento completato.")

## Andamento della loss del Critico

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(range(1, num_epochs + 1), critic_losses)
plt.xlabel("Epoca")
plt.ylabel("Loss Critico")
plt.title("Andamento della loss del Critico")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("risultati/loss_critico.png", dpi=150)
plt.show()

## Nota conclusiva

Il modello è stato addestrato con learning rate 0.0001, embedding di dimensione 50 e coefficiente Gradient Penalty pari a 10. Il Critico è stato aggiornato cinque volte per ogni aggiornamento del Generatore. Le gallerie permettono di confrontare l'evoluzione delle immagini dall'inizio alla fine del training.